<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/scale_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

Mounted at /content/drive
folder ready


In [3]:
%%writefile /content/drive/MyDrive/ml_project/scale_data.py
# -*- coding: utf-8 -*-
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from sklearn.preprocessing import StandardScaler

target_dir = "/content/drive/MyDrive/ml_project"

input_X = os.path.join(target_dir, "X.pkl")                 # wejście
choice_file = os.path.join(target_dir, "scale_method.txt")  # zapis wybranych kolumn
output_scaled = os.path.join(target_dir, "X_scaled.pkl")    # wyjście

# ---------------------------------------------------------
# scale_data_ui
# ---------------------------------------------------------
def scale_data_ui():

    if not os.path.exists(input_X):
        raise FileNotFoundError("X.pkl not found")

    X = pd.read_pickle(input_X)

    # wybieramy tylko kolumny numeryczne
    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]

    label = widgets.Label("select numerical columns to scale:")
    select = widgets.SelectMultiple(options=num_cols)

    btn = widgets.Button(description="confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        chosen_cols = list(select.value)

        with open(choice_file, "w") as f:
            f.write(",".join(chosen_cols) + "\n")

        with out:
            print("saved:", choice_file)
            print("columns to scale:", chosen_cols)

    btn.on_click(on_click)

    display(widgets.VBox([
        label,
        select,
        btn,
        out
    ]))

# ---------------------------------------------------------
# scale_data
# ---------------------------------------------------------
def scale_data():

    if not os.path.exists(input_X):
        raise FileNotFoundError("X.pkl not found")

    if not os.path.exists(choice_file):
        raise FileNotFoundError("scale_method.txt not found")

    X = pd.read_pickle(input_X)

    # odczyt kolumn do skalowania
    with open(choice_file, "r") as f:
        cols_raw = f.read().strip()

    cols = [c.strip() for c in cols_raw.split(",") if c.strip()]

    if len(cols) == 0:
        print("no columns selected for scaling")
        X.to_pickle(output_scaled)
        return X

    # skalowanie
    scaler = StandardScaler()

    X_scaled = X.copy()
    X_scaled[cols] = scaler.fit_transform(X[cols])

    # zapis wyniku
    X_scaled.to_pickle(output_scaled)

    print("saved:", output_scaled)
    print("scaled columns:", cols)

    return X_scaled

Writing /content/drive/MyDrive/ml_project/scale_data.py
